# Relationships between variables

*Correlation, crosstabs, pivot tables, and Simpson's paradox*

In [Chapter 4](ch-04-single-variable.qmd) we described one variable at a time. The questions that follow concern pairs of variables: do customers who order more also spend more per order? Do deeper discounts accompany bigger orders? Does the answer to these questions change across channels? In this chapter we construct the toolset for such questions: scatter plots, correlation, crosstabs, and pivot tables. We end with a paradox in Prairie Wholesale's discount data. Because of the paradox, we revise a pricing decision.

> **Setup for this chapter**
>
> You can run this chapter's notebook in two ways. In the cloud, [**open ch-05-relationships.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-05-relationships.ipynb); nothing needs to be installed. Locally, use the `pyba-core` environment (Appendix A). Either way, run the setup cell below first. No keys or paid accounts are needed anywhere in this book; Colab requires only a free Google account.

In [ ]:
# Setup. Run this cell once per session. It installs this chapter's
# packages; on Google Colab it also fetches the course data.
%pip install -q pandas plotly statsmodels
import sys
if "google.colab" in sys.modules:
    !git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

## Two numeric variables: the scatter plot {#sec-ch5-intro}

The **customer snapshot** (Appendix C) summarizes each customer's history as of a fixed date, in one row per active customer: order counts, revenue, discounts, and channel mix. A summary table of this kind is called a snapshot because it records the state of the customer base at one moment, the way a photograph records a scene. We load the snapshot and ask a first question: do customers who order more also spend more?

In [ ]:
import pandas as pd
import plotly.express as px

from pyba import DATA_DIR

snap = pd.read_csv(DATA_DIR / "pw_customer_snapshot.csv")
orders = pd.read_csv(DATA_DIR / "pw_orders.csv", parse_dates=["order_date"])
snap[["orders_12m", "revenue_12m", "avg_order_value", "pct_online"]].describe().round(2)

In [ ]:
px.scatter(snap, x="orders_12m", y="revenue_12m",
           hover_data=["business_type"], opacity=0.5,
           labels={"orders_12m": "Orders (12 months)",
                   "revenue_12m": "Revenue (12 months, $)"})

A **scatter plot** places one point per observation, at the observation's (x, y) values; nothing is summarized or fitted. The pattern here is positive and moderate (r = 0.54). Here r is the correlation coefficient, a measure of linear association on a scale from −1 to +1; we review it in the next section. The direction is unsurprising, because each order adds revenue. The informative points are the ones that deviate from the pattern. Points far above the trend are customers who place few but very large orders; the hover labels show that these are mostly schools. Points below the trend are customers who order often and in small amounts. From the scatter plot we see two types of good customer that a single summary would have combined.

## Correlation: the pattern as one number

The **correlation coefficient** (Pearson's *r*) summarizes a scatter plot's linear relationship in one number between −1 and +1. Its sign shows the direction of the association, and its magnitude shows the strength. An r of zero indicates no linear relationship.

In [ ]:
snap["orders_12m"].corr(snap["revenue_12m"]).round(3)

On DataFrames with more than two variables, `corr` returns the full matrix of pairwise correlations, which we can plot as a heatmap:

In [ ]:
num_cols = ["orders_12m", "revenue_12m", "avg_order_value",
            "avg_discount_pct", "pct_online", "weeks_since_last_order",
            "tenure_months"]

corr = snap[num_cols].corr().round(2)
px.imshow(corr, text_auto=True, color_continuous_scale="RdBu",
          zmin=-1, zmax=1, aspect="auto")

We read a correlation matrix in two passes. First we find the cells with large magnitudes. Then we ask what causes each one. Some correlations are mechanical: revenue correlates with order count because revenue is a sum over orders. Others describe the business and require an explanation. Online share and average discount are strongly negatively correlated (r = −0.72). We give the reason in the discount section below: reps approve the deepest discounts, and rep-heavy customers are the least online. The same variables return in [Chapter 9](../part-04-predictive/ch-09-classification-1.qmd), as the features of a churn model.

> **Python note: numeric_only**
>
> `snap.corr()` fails on the full table, since business type and region are text. Either select the numeric columns first, as above, or pass `snap.corr(numeric_only=True)`. The same argument is available on `mean`, `sum`, and the other aggregation methods, every time you apply a numeric operation to a mixed table.

### Calibrating what a given r looks like

Guessing a correlation from a scatter plot is difficult without training. An *r* of 0.6 sounds strong. The scatter plot behind that number looks looser than one would expect. To calibrate your eye, spend ten minutes with the game below.

::: {.content-visible when-format="html"}
<iframe src="../../assets/demos/correlation-game.html" width="100%" height="470" style="border:1px solid #d0d7de; border-radius:8px;" title="Guess the correlation game"></iframe>
:::

::: {.content-visible unless-format="html"}
The online version has a guessing game here: a random scatter plot is shown, you guess its r on a slider, and the true value is then revealed. Without training, you may over-read weak patterns and under-read strong ones.
:::

## The limits of correlation

Pearson's *r* measures **linear** association, so it can understate a strong pattern that is not a straight line. Extreme values influence it for the same reason they influence the mean. Both failure modes, the missed nonlinear pattern and the influential extreme value, can be observed in a scatter plot. As such, we always pair the number with the plot.

::: {.content-visible when-format="html"}
The four datasets below, a famous construction by the statistician Francis Anscombe, have the same r of 0.82 and the same fitted line. Click through them.

<iframe src="../../assets/demos/anscombe-r.html" width="100%" height="520" style="border:1px solid #d0d7de; border-radius:8px;" title="Anscombe's quartet"></iframe>
:::

::: {.content-visible unless-format="html"}
The online version has Anscombe's quartet here: four datasets that have the same r of 0.82 and the same fitted line yet completely different structures, switchable by a button.
:::


The quartet compresses the failure modes into one artificial example. We now observe them in Prairie Wholesale's own data.

### Curvature

Consider order volume against month of the year for school customers. August and January are far higher than the summer months, a strongly seasonal pattern. The correlation between order count and month number is nevertheless only −0.25, because the pattern rises and falls within the year and can't be represented by a straight line.

In [ ]:
schools = pd.read_csv(DATA_DIR / "pw_customers.csv")
schools = schools[schools["business_type"] == "school"]["customer_id"]

school_orders = orders[orders["customer_id"].isin(schools)]
by_month = (school_orders.groupby(school_orders["order_date"].dt.month)
            .size().rename("orders").rename_axis("month").reset_index())

by_month["orders"].corr(by_month["month"]).round(3)

In [ ]:
px.bar(by_month, x="month", y="orders",
       labels={"month": "Month", "orders": "School orders"})

### Outliers

One very large point can produce a correlation where there is none, or mask a true relationship. The school-district observation in [Chapter 4](ch-04-single-variable.qmd) is far from the rest of the data in every scatter plot it is part of. Any correlation calculated with that observation should be calculated once again without it. In [Chapter 8](../part-03-statistical/ch-08-regression.qmd) we quantify how much such a point can move a fitted line.

### Causation

Correlation is evidence that two variables move together. It carries no information about why. All the data in this book so far is **observational**: it records what customers did on their own, with nothing assigned or changed by the analyst. In the next section we work through a case in which a pricing decision depends on the distinction between correlation and causation.

## Categorical pairs: crosstabs and pivot tables

For two categorical variables we use a **crosstab**: a grid of counts, with one variable's categories as the rows and the other's as the columns. Which business types use which channels?

In [ ]:
lines = orders.merge(pd.read_csv(DATA_DIR / "pw_customers.csv")[["customer_id", "business_type"]],
                     on="customer_id", how="left", validate="many_to_one")

pd.crosstab(lines["business_type"], lines["channel"], normalize="index").round(2)

`normalize="index"` converts each row into shares. Shares are the quantity of interest here: within each business type, how do its orders split across channels? Offices and clinics order the most online. Schools order mostly by phone. They also use reps more than any other business type does. For the numeric version of the same question, we use a **pivot table**. The grid is the same, but each cell now holds a summary computed from the rows that fall in that cell: here, the mean of `line_total` for one business type ordering through one channel.

In [ ]:
pivot = lines.pivot_table(values="line_total", index="business_type",
                          columns="channel", aggfunc="mean")
pivot.round(2)

In [ ]:
px.imshow(pivot.round(0), text_auto=True, color_continuous_scale="Blues", aspect="auto")

We carry one fact from the heatmap into the next section: the rep channel's lines are larger for every business type.

## Simpson's paradox in the discount column

Prairie Wholesale's sales director wonders whether larger discounts are associated with larger orders. Aggregating all orders, we find the correlation between an order's discount rate and its total:

In [ ]:
order_totals = orders.groupby("order_id").agg(
    total=("line_total", "sum"),
    discount=("discount_pct", "first"),
    channel=("channel", "first"),
)

order_totals["discount"].corr(order_totals["total"]).round(3)

An estimate computed over all orders together, with no attention to subgroups, is called **pooled**. The pooled correlation here is positive: deeper discounts go with bigger orders. From the pooled number alone, we might conclude that discounts increase order size and that the company should therefore discount more. Now we compute the same correlation within each channel:

In [ ]:
order_totals.groupby("channel").apply(
    lambda g: g["discount"].corr(g["total"]), include_groups=False
).round(3)

The within-channel correlations are negative in all three channels: within online, phone, and rep orders alike, deeper discounts go with smaller totals. A pooled estimate whose sign reverses in every subgroup is an instance of **Simpson's paradox**, a mathematical possibility whenever a third variable (here, the channel) is associated with both of the variables under study.

We can see the mechanism in the pieces we already have. Rep-negotiated orders are much larger than online and phone orders. Reps also authorize the deepest discounts. As such, the pooled data contains a group of large, heavily discounted rep orders. The line fitted to all orders together therefore slopes upward. Within any one channel, the ordinary logic is visible: the most discounted orders are the smaller, price-driven ones.

In [ ]:
# trendline="ols" fits the lines with statsmodels (introduced in Chapter 8)
sample = order_totals[order_totals["total"] < 2000].sample(1200, random_state=0)

fig = px.scatter(sample, x="discount", y="total", color="channel",
                 log_y=True, opacity=0.4, trendline="ols",
                 color_discrete_map={"online": "#0969da", "phone": "#bf8700", "rep": "#8250df"},
                 labels={"discount": "Discount rate", "total": "Order total ($, log)"})
pooled = px.scatter(sample, x="discount", y="total", log_y=True,
                    trendline="ols", trendline_color_override="#57606a")
fig.add_trace(pooled.data[1])
fig.show()

Both numbers are calculated correctly. The within-channel number is the answer to the director's question. The channel is a **confounder**: it is related to both the discount on an order and the size of the order. By holding the channel fixed, we remove its effect. We state the general principle as a rule: **before acting on any pooled correlation, re-calculate it within the natural subgroups of the business.** In Part III we convert this discipline into formal tools. The free-shipping pilot in @sec-ch7-intro exists because Prairie Wholesale wanted an answer that remains true after confounders are accounted for.

## Evaluation: is the claim true within the subgroups?

In this chapter we made a claim: "discounts and order size are positively related". We evaluate the claim by recomputing it within subgroups.

|  |  |
|---|---|
| **Metric** | the sign and size of *r*, pooled and within each channel. |
| **Test set** | all 40,000 orders, split by the grouping the business itself uses. |
| **Baseline** | the pooled estimate. |

: {tbl-colwidths="[18,82]"}

In [ ]:
rows = {"pooled": order_totals["discount"].corr(order_totals["total"])}
for ch, g in order_totals.groupby("channel"):
    rows[ch] = g["discount"].corr(g["total"])
pd.Series(rows).round(3).to_frame("r(discount, total)")

Within the subgroups the claim is false: the pooled sign reverses in every one. Therefore, the pooled number was an artifact of channel mix. A claim that remains true in this table is taken to Part III for a formal test; a claim contradicted here is not used for any decision-making purposes.

## The decision this informs

The sales director considered increasing discounts to increase order size. The subgroup analysis shows that the pooled evidence for this plan is an artifact of channel mix. Within each channel, the association runs in the opposite direction. The appropriate decision is to forgo an across-the-board discount expansion and to assess discount policy by channel. A separate margin review of the rep channel's deep discounts on already-large orders is warranted (in [Chapter 3](../part-01-foundations/ch-03-pandas.qmd) we calculated what discounts cost, by category). To establish the causal effect of discounts on order size, the company must run an experiment. We begin Part III with one such experiment.

## Exercises



### Build lab

Investigate the value of *tenure*: is `tenure_months` associated with `revenue_12m` in the snapshot? Produce the scatter plot (consider a log y-axis), the correlation, and then the within-group correlations by `business_type`. Summarize in two sentences: what is the pooled association, and is it still present within the subgroups?

### Evaluate lab

Take the strongest off-diagonal cell you can find in the correlation matrix (@fig-corr-matrix), excluding the two mechanical revenue pairs (revenue with orders, revenue with average order value) and the online/discount pair we explained above. Evaluate it the way we evaluated the discount claim: recompute it within business types, present the table, and give a verdict in one sentence: is the cell an artifact of a confounder or a robust association?